# Gradient descent in neural networks: batch, stochastic and mini-batch
The same small network trained on the Social Network Ads data with different `batch_size` values: time per run,
progress per epoch, the shape of the loss curve, and how Keras cuts the rows into batches.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import time
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.config.experimental.enable_op_determinism()   # same numbers on every run
tf.get_logger().setLevel("ERROR")                # hide TensorFlow's retracing notes

In [ ]:
df = pd.read_csv("data/Social_Network_Ads.csv")[["Age", "EstimatedSalary", "Purchased"]]
X = StandardScaler().fit_transform(df[["Age", "EstimatedSalary"]])   # very different scales -> mean 0, std 1
y = df["Purchased"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
print(df.shape, X_train.shape)

In [ ]:
def build(seed=0):
    keras.utils.set_random_seed(seed)        # same starting weights every time
    model = keras.Sequential([keras.Input(shape=(2,)),
                              keras.layers.Dense(10, activation="relu"),
                              keras.layers.Dense(10, activation="relu"),
                              keras.layers.Dense(1, activation="sigmoid")])
    # plain gradient descent, so batch_size alone decides batch / stochastic / mini-batch
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

class CountUpdates(keras.callbacks.Callback):
    # counts the weight updates (batches) in each epoch
    def on_epoch_begin(self, epoch, logs=None):
        self.n = 0
    def on_train_batch_end(self, batch, logs=None):
        self.n += 1

build().summary()

## Same number of epochs: which finishes first?

In [ ]:
build().fit(X_train, y_train, epochs=1, verbose=0)    # warm-up: the first fit also sets TensorFlow up
timing = []
for bs in [320, 32, 1]:
    m, cb = build(), CountUpdates()
    start = time.time()
    m.fit(X_train, y_train, epochs=10, batch_size=bs, callbacks=[cb], verbose=0)
    secs = time.time() - start
    timing.append(dict(batch_size=bs, updates_per_epoch=cb.n, seconds=round(secs, 2)))
pd.DataFrame(timing)

## Same number of epochs: which gets closer to a solution?
All 400 rows, the last 20% kept for validation, 10 epochs.

In [ ]:
progress = []
for bs in [320, 32, 1]:
    h = build().fit(X, y, epochs=10, batch_size=bs, validation_split=0.2, verbose=0)
    progress.append(dict(batch_size=bs, val_accuracy=round(h.history["val_accuracy"][-1], 3),
                         loss=round(h.history["loss"][-1], 3)))
pd.DataFrame(progress)

## The loss after every update: smooth or jagged?

In [ ]:
curves = {}
for bs in [320, 32, 1]:
    m = build()
    h = m.fit(X_train, y_train, epochs=100, batch_size=bs, verbose=0)
    curves[f"batch_size={bs}"] = h.history["loss"]
    print(bs, "loss after 100 epochs:", round(h.history["loss"][-1], 3),
          "test accuracy:", round(m.evaluate(X_test, y_test, verbose=0)[1], 3))
pd.DataFrame(curves).rename_axis("epoch").to_csv("data/epoch_loss.csv")

Within one epoch of stochastic gradient descent, the training loss after each single update:

In [ ]:
def per_update_loss(bs, n_updates=320):
    m, out = build(), []
    rng = np.random.default_rng(0)
    order = rng.permutation(len(X_train))
    for start in range(0, n_updates * bs, bs):
        idx = order[np.arange(start, start + bs) % len(X_train)]
        m.train_on_batch(X_train[idx], y_train[idx])
        p = np.clip(m(X_train, training=False).numpy().ravel(), 1e-7, 1 - 1e-7)
        # binary cross-entropy on all 320 training rows after this update
        out.append(float(-np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))))
    return out

steps = pd.DataFrame({"sgd (batch_size=1)": per_update_loss(1, 320),
                      "mini-batch (batch_size=32)": per_update_loss(32, 320)})
steps.rename_axis("update").to_csv("data/update_loss.csv")
steps.describe().loc[["min", "max"]]

## When batch_size does not divide the rows

In [ ]:
for bs in [150, 250, 400]:
    cb = CountUpdates()
    build().fit(X, y, epochs=1, batch_size=bs, callbacks=[cb], verbose=0)
    n = len(X)
    print(f"400 rows, batch_size={bs}: {cb.n} updates per epoch; batch sizes",
          [min(bs, n - i) for i in range(0, n, bs)])
print("Keras default batch_size (when none is given) is 32:", int(np.ceil(400 / 32)), "updates per epoch")